In [ ]:
## EXPERIMENT -2
# with  TRAINROWS = 45000 ( Ring analyis -As GPU has the memory limit)
# My experiment - mini batch neighbor sampling + Hybrid model (Hypergraph NN + Heterogeneous Graph Transformers (HGTConv)) for large scale Fraud detection ##
### Adding rolling behavioral features + Temporal deltas
## Comapred with XGBOOST



import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HGTConv    #### Heterogeneous Graph Transformers (HGTConv)
from torch_geometric.loader import NeighborLoader 
import math

from torch_geometric.nn import Linear
from torch.utils.data import Dataset, DataLoader
from pyvis.network import Network
from dataclasses import dataclass
from collections import defaultdict

from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ( average_precision_score, confusion_matrix, accuracy_score)
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import pandas as pdt
import numpy as np
import os
import gc
import time
import matplotlib.pyplot as plt
import networkx as nx
import random
from IPython.display import display, HTML


try:
    import xgboost as xgb
except Exception:
    xgb = None

try:
    import seaborn as sns
except ImportError:
    sns = None  

try:
    from scipy.stats import mannwhitneyu, spearmanr
except Exception:
    mannwhitneyu = None
    spearmanr = None   



DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.cuda.empty_cache()  ## clear GPU cache
### Feature dims and training
ENTITYDIM = 32
TXN_FEAT_DIM = 6
TIME_DIM = 12  ### Dimention of Time2Vec encoding
HIDDEN_DIM = 128
OUT_EMB_DIM = 64
LR = 5e-4
EPOCHS = 20
SEED = 45
TRAINROWS = 45000 ## if we select None here, then that would consider full dataset.
GRAPH_HOPS_FOR_RING = 2
RING_TXN_ID = 1234
THRESH =  0.80  # 0.7  #0.9
use_amp: bool = True
train_frac_time: float = 0.8
out_ablation_csv: str ="ablationresults.csv"
out_ring_metrics_csv: str = "ring_metrics.csv"
out_queue_csv: str = "investigation_queue.csv"
out_budget_csv: str ="budget_compare.csv"
out_adaptive_budget_csv: str="Myadaptivebudgetlearnedalpha.csv"
###Regularization
WEIGHT_DECAY = 1e-4
###Overfitting control validation based on early stopping###
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4
RUN_RING_STAGE: bool = True
RING_TOPK_FLAGGED:  int = 300  #2000
RING_HOPS:  int = 1  #2  #1
RING_ALPHA: float= 0.7  #0.7   
#RING_COMBINE_MODE: str ="linear"
RING_COMBINE_MODE: str ="rank"

RING_BUDGETS: tuple =(25, 50, 100, 200, 500)
RING_CAUSAL: bool = True
##Time split
train_val_frac_time = 0.8
val_frac_within_train = 0.1

POS_WEIGHT = None
RINGTHRESH = 0.4
ALERTTHRESH = 0.7



### mini batch training - Neighbor sampling
BATCH_SIZE: int = 512 # batch size for neighbor sampling on hetero-graph
NEIGHBORS: tuple[int, int] = [15, 10]  #number of neighbors to sample at 1-hop and 2-hop for HGTConv( for a 2 layer GNN)

## Reproducibility
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

##Reduce Memory usage here ## Optimize Data types
def reduce_mem_usage(df):
     for col in df.columns:
         if not pd.api.types.is_object_dtype(df[col]):
             df[col] = df[col].astype(np.float32)
     print("Memory reduce")
     return df
        
def normalizeArray(a):
    a = a.astype(np.float32)
    return (a - a.min()) / (a.max() - a.min() +1e-6)

def compute_metrics_updated(y_true, y_prob, threshold = THRESH):
    ### if threshold is None then I will calculate only AUC 
    ### else ACU + all threshold metrics will be calculated here.####
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_prob = np.asarray(y_prob).astype(float).reshape(-1)
    metrics ={"acc": None, "prec": None, "rec": None , "f1": None, "auc": 0.0, "pr_auc": 0.0}
    if y_true.size == 0:
        return metrics
    
    try:
        metrics["auc"]= float(roc_auc_score(y_true,y_prob))
    except Exception:
        metrics["auc"] = 0.0

    try:
        metrics["pr_auc"] = float(average_precision_score(y_true, y_prob))
    except Exception:
        metrics["pr_auc"] = 0.0

    print("threshold value", threshold)
    if threshold is not None:
        y_pred= (y_prob >= threshold).astype(int)
        metrics["acc"] = float(accuracy_score(y_true, y_pred))
        metrics["prec"] = float(precision_score(y_true, y_pred, zero_division = 0))
        metrics["rec"] = float(recall_score(y_true, y_pred, zero_division = 0))
        metrics["f1"] = float(f1_score(y_true, y_pred, zero_division = 0))

   
    return metrics
        
def plot_confusion_matrix(y_true, y_prob, threshold = THRESH, title = "Confusion Matrix"):
    print("inside plot_confusion_matrix")
    if threshold is None:
        y_pred = np.asarray(y_prob).astype(int)
        #print("threshold is None , we are skipping plot confusion matrix")
        #return
    else:
         y_pred = (np.asarray(y_prob) >= threshold).astype(int)

    y_true = np.asarray(y_true).astype(int)
    #y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    cm = confusion_matrix(y_true , y_pred)
    plt.figure(figsize = (5,4))
    if sns is not None:
        sns.heatmap(cm, annot = True, fmt ="d", cmap ="Blues", cbar = False)
    else:
        plt.imshow(cm, cmap ="Blues")
        for(i, j), v in np.ndenumerate(cm):
            plt.text(j, i, str(v), ha="center", va = "center")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show() 


def roleassign(indg, outdg):
    if outdg > 2 * indg:
        return "SOURCE"
    if indg > 2 * outdg:
        return "SINK"
    if indg +outdg > 10:
        return "HUB"
    return "PERIPHERAL"



def make_time_split_train_val_test(df: pd.DataFrame, time_col ="TransactionDT",
                                   train_val_frac =0.8, val_within_train =0.1):
    t = df[time_col].values.astype(np.float64)

    p_val_end = float(train_val_frac)
    p_train_end = float(train_val_frac * (1.0 - val_within_train))
    p_train_end = max(0.0, min(p_train_end, p_val_end))

    cutoff_train = np.quantile(t, p_train_end)
    cutoff_val = np.quantile(t, p_val_end)

    train_mask = t <= cutoff_train
    val_mask =(t > cutoff_train) & (t <= cutoff_val) 
    test_mask = t > cutoff_val

    train_idx = torch.tensor(np.where(train_mask)[0], dtype=torch.long)
    val_idx = torch.tensor(np.where(val_mask)[0], dtype=torch.long)
    test_idx = torch.tensor(np.where(test_mask)[0], dtype=torch.long)


    return  train_idx, val_idx, test_idx, cutoff_train, cutoff_val
 



def _shannon_entropy_from_counts(counts: np.ndarray) -> float:
    counts = np.asarray(counts, dtype = np.float64)
    s = counts.sum()
    if s <= 0:
        return 0.0
    p = counts / s
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

#### Time based split. Train past + test future data

def make_time_split_indices(df: pd.DataFrame, time_col="TransactionDT", train_frac=0.8):
    t = df[time_col].values.astype(np.float64)
    cutoff = np.quantile(t, train_frac)
    train_mask = t <=  cutoff
    test_mask = ~train_mask
    train_idx = torch.tensor(np.where(train_mask)[0], dtype = torch.long)
    test_idx = torch.tensor(np.where(test_mask)[0], dtype = torch.long)
    return train_idx, test_idx, cutoff



## Temporal Encoding ##
## vector representation for a scalar time input. This class will help to capture evolving fraud patterns.##
class Time2Vector(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.w = nn.Parameter(torch.randn(1, out_dim))
        self.b = nn.Parameter(torch.randn(1, out_dim))
                
    def forward(self, t):   ##t: shape[N, 1] time value per sample
        return torch.sin(t * self.w  + self.b)   #### Output: shape[N, TIME_DIM] with sinusoidal time encoding
        
class TemporalAttention(nn.Module):
    def __init__(self, time_dim):
        super().__init__()
        self.att = nn.Linear(time_dim, time_dim, bias = False)  ## Learn wiights to score each time component

    def forward(self, time_emb):
        scores = self.att(time_emb)
        weights = torch.softmax(scores, dim =1)  ## attention weights across time dimensions
        return time_emb * weights, weights

   
###########visualization#######
def plot_time2vec(time2Vec, t_min, t_max, steps =500):
    t = torch.linspace(t_min, t_max, steps).reshape(-1, 1).to(DEVICE)
    with torch.no_grad():
        v = time2Vec(t).cpu().numpy()

    plt.figure(figsize = (12,4))
    for i in range(v.shape[1]):
        plt.plot(v[:, i], label = f"dim{i}")
    
    plt.title("Time2Vector Temporal Embeddings")
    plt.xlabel("Time")
    plt.ylabel("Activation")
    plt.legend(ncol = TIME_DIM, fontsize = 7)
    plt.show()


############ FRAUD and Non Fraud temporal attention ###

def plot_temporal_attention(fraud_att, nonfraud_att):
    plt.figure(figsize = (10,4))
    plt.plot(fraud_att.mean(axis = 0), marker = "o", label ="Fraud")
    plt.plot(nonfraud_att.mean(axis = 0), marker = "o", label ="Non-Fraud")
    plt.title("Temporal Attension : which Time Patterns Matter?")
    plt.legend()
    plt.grid(True)
    plt.show()

    
def combineScores(prob: np.ndarray, ring_score: np.ndarray, alpha: float = RING_ALPHA, mode: str = RING_COMBINE_MODE) -> np.ndarray:
    prob = np.asarray(prob, dtype=np.float32)
    ring_score = np.asarray(ring_score, dtype=np.float32)

    if mode.lower() == "linear":
        return alpha * prob + (1.0 - alpha) * ring_score
    
      
    if mode.lower() == "rank":
        pr = prob.argsort().argsort().astype(np.float32)
        rr = ring_score.argsort().argsort().astype(np.float32)
        pr = pr / (max(1, len(pr) - 1))
        rr  = rr / (max(1, len(rr) -1))
        return  alpha * pr + (1.0 - alpha) * rr
    
    if mode.lower() == "zscore":
        def z(x):
            x = x.astype(np.float32)
            return (x - x.mean()) / (x.std() + 1e-6)
        return alpha * z(prob)  + (1.0 - alpha) * z(ring_score)
    
    # fallback
    return alpha * prob + (1.0 - alpha) * ring_score

def combineMytrustscore(prob, ringscore, trust, alp0=RING_ALPHA, gama=0.5):
    ## if trust is low at the time of checking , then analysis will depends on model probability. 
    ## if trust is showing high value then allow ring score to influence more.
    prob = np.asarray(prob, dtype=np.float32)
    ringscore = np.asarray(ringscore, dtype=np.float32)
    trust = np.asarray(trust, dtype=np.float32)
    alphai = np.clip(alp0 +gama * (1.0 - trust), 0.0, 1.0)
    return alphai * prob + (1.0 - alphai) * ringscore


def build_investigation_queue(df_ring: pd.DataFrame, budgets: tuple = RING_BUDGETS) -> tuple[pd.DataFrame, pd.DataFrame]:
    if df_ring is None or df_ring.empty:
        return None, None
    
    ring_col = None
    if "ring_consistency_score" in df_ring.columns:
        ring_col = "ring_consistency_score"
    elif "ring_consistency_score" in df_ring.columns:
        ring_col = "ring_consistency_score"  

    if ring_col is None:
        print("No ring score column found ") 
        return None, None  
    
    reqd1 = ["txn_id", "prob", "label", ring_col]
    misn1 = [c for c in reqd1 if c not in df_ring.columns]
    if misn1:
        raise ValueError(f"missing col: {misn1}")

    d = df_ring.copy()

    def _minmax(x: np.ndarray) -> np.ndarray:
        x = np.asarray(x, dtype= np.float64)
        x = np.nan_to_num(x , nan = 0.0, posinf = 0.0 , neginf = 0.0)
        lo, hi = float(x.min()) , float(x.max())
        if hi - lo < 1e-12:
            return np.zeros_like(x, dtype=np.float32)
        a = x - lo
        b = hi - lo
        return (a/ b).astype(np.float32)

    for c in ["prob", "label", "ring_consistency_score"]:
         if c not in d.columns:
            raise ValueError(f"df_ring missing required colum: {c}")
    d["score_model"] = _minmax(d["prob"].values.astype(np.float32))
    d["score_ring"] = _minmax(d[ring_col].values.astype(np.float32))
    d["score_combined"] = combineScores(d["score_model"].values, d["score_ring"].values, alpha=RING_ALPHA, mode=RING_COMBINE_MODE).astype(np.float32)
    q_model = d.sort_values("score_model", ascending = False).reset_index(drop = True).copy()
    q_ring = d.sort_values("score_ring", ascending = False).reset_index(drop=True).copy()
    q_comb = d.sort_values("score_combined", ascending = False).reset_index(drop=True).copy()

    q_model["rank_model"] = np.arange(1, len(q_model) + 1)
    q_ring["rank_ring"] = np.arange(1, len(q_ring) + 1)
    q_comb["rank_combined"] = np.arange(1, len(q_comb) + 1)

    queue_df = (
        q_comb[["txn_id","prob","label",ring_col,"score_model","score_ring", "score_combined", "rank_combined"]]
        .rename(columns ={ring_col: "ring_consistency_score"})
        .merge(q_model[["txn_id", "rank_model"]], on ="txn_id", how="left")
        .merge(q_ring[["txn_id", "rank_ring"]], on ="txn_id", how ="left")

    )


    def prec_at(q: pd.DataFrame, B: int) -> float:
        top =q.head(int(B))
        if len(top) == 0:
            return 0.0
        return float((top["label"].values.astype(int) == 1).mean())
    
    total_pos = int((d["label"].values.astype(int) == 1).sum())
    def recall_at(q: pd.DataFrame, B: int) -> float:
        if total_pos <= 0:
            return 0.0
        top = q.head(int(B))
        return float((top["label"].values.astype(int)  == 1).sum() / total_pos)
    
    rows = []
    for B in budgets:
        B = int(B)
        rows.append({
            "budget": B,
            "precision@B_model": prec_at(q_model, B),
            "precision@B_ring": prec_at(q_ring, B),
            "precision@B_combined": prec_at(q_comb, B),
            "recall@B_model": recall_at(q_model, B),
            "recall@B_ring": recall_at(q_ring, B),
            "recall@B_combined": recall_at(q_comb, B) })
        
    comp_df = pd.DataFrame(rows)
    return queue_df, comp_df



### My step to build Heterogeneous Hypergraph############ 

##data = HeteroData()

###TO Support millions transactions Fraud detection## trying to build true hypergraph

#### HYPERGRAPH MODEL (fixed- k incidence#########################

class DualHypergraph:
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop = True).copy()
        self.N = len(self.df)
        s11 = self.N
        print("selfn size : ", s11)
         ### Define attributes that will form hyperedges (including composite attributes)
        self.df["time_bin"] = (self.df["TransactionDT"].astype(np.int64) // (60 * 5)).astype(str)  ### 5 minute time bins
            #hyper_attrs = ["card1","ProductCD","addr1","P_emaildomain"]
        ###priority fraud attributes at table

        base_cols = ["card1","addr1","P_emaildomain","DeviceInfo","DeviceType","id_30","id_31","ProductCD", "time_bin"]

        print("Column seen by Hypergraph:", sorted(self.df.columns))

        self.df["combo_dev_ip"] = self.df["DeviceInfo"].astype(str) +"_"+ self.df["id_31"].astype(str)
        self.df["combo_card_addr"] = self.df["card1"].astype(str) +"_"+ self.df["addr1"].astype(str)
        #df["combo_email_addr"] = df["P_emaildomain"].astype(str) +"_"+ df["addr1"].astype(str)
        self.attr_cols = base_cols + ["combo_dev_ip", "combo_card_addr"]

        for c in self.attr_cols:
            self.df[c] = self.df[c].fillna("unknown").astype(str)

        #self.df = df 
        offsets = {}
        total = 0
        codes_per_col = []

        #self.num_nodes = len(self.df)
        #self.hyperedge_offsets = {}
        #self.value_maps = {}

        ###build hyperedge index spaces
        for col in self.attr_cols:
            cat = self.df[col].astype("category")
            codes = cat.cat.codes.astype(np.int32)
            #uniqueVal = pd.Series(self.df[col].unique()).astype(str)
            m = int(codes.max()) + 1 if len(codes) else 0
            offsets[col] = total
            total += m
            codes_per_col.append(codes)

        self.total_hyperedges = int(total)
        self.K = len(self.attr_cols)
        he_cols = []
        for col, codes in zip(self.attr_cols, codes_per_col):
            he_cols.append((codes + offsets[col]).astype(np.int32))

        he_ids_mat = np.stack(he_cols, axis =1) # shape [N,K]  , N= number of transactions and K = number of hyperedges per transaction(fixed)

        self.he_ids_mat = torch.from_numpy(he_ids_mat)  #CPU int32
        try:
            self.he_ids_mat = self.he_ids_mat.pin_memory()
        except Exception:
            pass    

        self.labels = torch.tensor(self.df["isFraud"].fillna(0).values.astype(np.int64))
        mem_mb = self.he_ids_mat.numel() * 4 / 1e6 
        ###numel is number of elements in the tensor . 4 means 4 bytes per element. 
        ### 1e6 = 1,000,000  it converts bytes -> megabytes(MB)
        ## the line estimates RAM footprint of the hypergraph incidence matrix. its a sanity check, not an exact measurement.

        #### Hypergraph incidence storage is O(N * K)

        print(f"[HyperCSR] N={self.N:,} K={self.K} total_hyperedges={self.total_hyperedges:,} he_ids_mat={mem_mb:.1f}MB")


    def get_batch_hyperedges(self, batch_nodes: torch.Tensor, device: torch.device):
        bn = batch_nodes.detach()
        if bn.is_cuda:
            bn = bn.cpu()
        he = self.he_ids_mat.index_select(0, bn.long())  #CPU[B,K]
        return he.to(device=device, dtype=torch.long, non_blocking=True)     

      

                
class HypergraphNN(nn.Module):
    def __init__(self, num_nodes, num_hyperedges, K, dim =ENTITYDIM , hidden = HIDDEN_DIM, heads = 4):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, dim)  #Embedding for transaction
        self.he_emb = nn.Embedding(num_hyperedges, dim)  # Embeding for hyperedges
        self.proj = nn.Linear(dim * 2, hidden)
        #self.att = nn.MultiheadAttention(hidden, heads, batch_first = True)
        self.attn = nn.MultiheadAttention(embed_dim = hidden, num_heads=heads, batch_first = True)
        #self.K = K
        self.out_ln = nn.LayerNorm(hidden)

    def forward(self, batch_nodes, he_ids):
        ne = self.node_emb(batch_nodes)
        he = self.he_emb(he_ids)
        ne_expand = ne.unsqueeze(1).expand(-1, he.size(1),-1)
        M = torch.cat([ne_expand, he], dim=-1)
        h = F.relu(self.proj(M))
        att_out, _ = self.attn(h,h,h)
        out = att_out.mean(dim =1)
        return self.out_ln(out)

         

################
# Heterogeneous graph builder
################
def build_hetero_graph(df: pd.DataFrame, tdt_min: float, tdt_max: float):
    print("Buid Heterogeneous Graph............")
    data = HeteroData()

    amt = df["TransactionAmt"].values.astype(np.float32)
    tdt = df["TransactionDT"].values.astype(np.float32)
    tdt_norm = (tdt - tdt_min) / (tdt_max - tdt_min + 1e-6)
    
    ## ADD  Temporal evolution features

    delta_t = df["delta_t"].values.astype(np.float32)
    txn_count_1h = df["txn_count_1h"].values.astype(np.float32)
    txn_count_24h = df["txn_count_24h"].values.astype(np.float32)
    amt_zscore = df["amt_zscore"].values.astype(np.float32)

     
    X = np.column_stack([amt, tdt_norm, delta_t,txn_count_1h, txn_count_24h, amt_zscore ])

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    data["transaction"].x = torch.tensor(X, dtype= torch.float32)
    data["transaction"].y = torch.tensor(df["isFraud"].values.astype(np.int64), dtype = torch.long)
    num_txn = data["transaction"].x.size(0)

    df["device_code"] = df["DeviceInfo"].astype("category").cat.codes.astype(np.int64)
    df["ip_code"] = df["id_31"].astype("category").cat.codes.astype(np.int64)
    df["merchant_code"] = df["ProductCD"].astype("category").cat.codes.astype(np.int64)
    df["location_code"] = df["addr1"].astype("category").cat.codes.astype(np.int64)

   
    num_devices = int(df["device_code"].max()) + 1
    num_ips = int(df["ip_code"].max()) + 1
    num_merchants =  int(df["merchant_code"].max()) + 1
    num_locations =  int(df["location_code"].max()) + 1

    data["device"].x = torch.arange(num_devices, dtype = torch.long)
    data["ip"].x = torch.arange(num_ips, dtype = torch.long)
    data["merchant"].x = torch.arange(num_merchants, dtype = torch.long)
    data["location"].x = torch.arange(num_locations, dtype = torch.long)

         
    ### build edge transaction -> device
    #### connect transaction -> device placeholder
    ti = torch.arange(num_txn, dtype = torch.long) 
    di = torch.tensor(df["device_code"].values, dtype = torch.long)
    ii = torch.tensor(df["ip_code"].values, dtype = torch.long)
    mi = torch.tensor(df["merchant_code"].values, dtype = torch.long)
    li = torch.tensor(df["location_code"].values, dtype = torch.long)


    data[("transaction" , "uses", "device")].edge_index = torch.stack([ti, di])
    data[("device" , "used_by", "transaction")].edge_index = torch.stack([di, ti])

    data["transaction", "originates", "ip"].edge_index = torch.stack([ti, ii])
    data["ip", "source_of", "transaction"].edge_index = torch.stack([ii, ti])
  
    data["transaction", "done_at", "merchant"].edge_index = torch.stack([ti, mi])
    data["merchant", "has_transaction", "transaction"].edge_index = torch.stack([mi, ti])
    
    data["transaction", "occutrs_in", "location"].edge_index = torch.stack([ti, li])
    data["location", "loc_of", "transaction"].edge_index = torch.stack([li, ti])

    
    node_sizes = {
        "transaction": num_txn,
        "device": num_devices,
        "ip": num_ips,
        "merchant": num_merchants,
        "location": num_locations
        
    }

    return data, node_sizes

def safecopy(full_data: HeteroData) -> HeteroData:
    data = HeteroData()

    for ntype in full_data.node_types:
        for key, val in full_data[ntype].items():
            if isinstance(val, torch.Tensor):
                data[ntype][key] = val.clone()
            else:
                data[ntype][key] = val

    for etype in full_data.edge_types:
        for key, val in full_data[etype].items():
            if isinstance(val, torch.Tensor):
                data[etype][key] = val.clone()
            else:
                data[etype][key] = val
    return data                          


def filter_hetero_for_training(full_data: HeteroData, train_idx: torch.Tensor) -> HeteroData:
    data = safecopy(full_data)
    #data = full_data.clone()

    print("Inside filter_hetero_for_training")
    train_idx_cpu = train_idx.cpu()

    for (src, rel, dst) in full_data.edge_types:
        ei = full_data[(src, rel, dst)].edge_index
        if ei is None or ei.numel() == 0:
            data[(src, rel, dst)].edge_index = ei.clone() if isinstance(ei, torch.Tensor) else ei
            continue
        if src == "transaction":
            keep = torch.isin(ei[0].cpu(), train_idx_cpu)
            data[(src, rel, dst)].edge_index = ei[:, keep.to(ei.device)]
        elif dst == "transaction":
            keep = torch.isin(ei[1].cpu(), train_idx_cpu)
            data[(src, rel, dst)].edge_index = ei[:, keep.to(ei.device)]
        else:
            data[(src, rel, dst)].edge_index = ei    

    print("end of hetero training******")
    return data
        


####train/ eval (ABLATIONS) #############

def make_loader(hetero_data: HeteroData, idx: torch.Tensor, shuffle = True):
    if len(idx) == 0:
        return None
    return NeighborLoader(
        hetero_data, 
        input_nodes = ("transaction", idx),
        num_neighbors = list(NEIGHBORS),
        batch_size = BATCH_SIZE,
        shuffle = shuffle
        )

def set_mode(comps, train: bool):
    for m in comps.values():
        if isinstance(m, nn.Module):
            m.train(train)

def forward_variant(comps, hg: DualHypergraph, batch, seed_nodes, seed_count, return_aux:bool=False):
    mlp = comps.get("mlp")
    hyp = comps.get("hyp")
    het = comps.get("het")
    head = comps.get("head")
    fusion = comps.get("fusion")
    time_enc = comps.get("time_enc")
    mlp_time = comps.get("mlp_time")
    global_time = comps.get("_global_time")

    if mlp is not None:
        x_seed = batch["transaction"].x[:seed_count].float().to(DEVICE)
        if mlp_time:
            t_raw = get_seed_time(batch, seed_count, global_time_tensor=global_time)
            #t_simple = torch.sin(t_raw)
            #x_seed = torch.cat([x_seed, t_simple], dim =1)
            x_seed = torch.cat([x_seed, t_raw], dim =1)
            print("end of my MLP test")
        return mlp(x_seed)
    
    if hyp is not None and het is None and fusion is None and head is not None:
        he_ids = hg.get_batch_hyperedges(seed_nodes, device=DEVICE)
        emb = hyp(seed_nodes, he_ids)
        return head(emb)
    if het is not None and hyp is None and fusion is None and head is not None:
        tx_emb_all = het(batch)
        emb = tx_emb_all[:seed_count]
        return head(emb)
    # Fusion
    if fusion is not None and hyp is not None and het is not None:
        he_ids = hg.get_batch_hyperedges(seed_nodes, device=DEVICE)
        hyper_emb = hyp(seed_nodes, he_ids)
        tx_emb_all = het(batch)
        hetero_emb = tx_emb_all[:seed_count]

        if return_aux:
            return fusion(hyper_emb, hetero_emb, return_aux = True)
        else:
            return fusion(hyper_emb, hetero_emb)
    
    if fusion is not None and hyp is not None and time_enc is not None and het is None:
        he_ids = hg.get_batch_hyperedges(seed_nodes, device=DEVICE)
        hyper_emb = hyp(seed_nodes, he_ids)

        tx_feats = batch["transaction"].x[:seed_count].float().to(DEVICE)

        t_raw = get_seed_time(batch, seed_count, global_time_tensor=getattr(time_enc, "_global_time", None))
        time_emb = time_enc(tx_feats, t_raw)
        if return_aux:
            return fusion(hyper_emb, time_emb, return_aux = True)
        return fusion(hyper_emb, time_emb)   
    
    raise ValueError("##Invalid components configuration in ablation variant")
    
    

def amp_ok_for_comps(comps) -> bool:
    if not (use_amp and DEVICE.type == "cuda"):
        return False
    if comps.get("het") is not None:
        return False
    #if comps.get("fusion") is not None and comps.get("het") is not None:
        #return False
    if comps.get("time_enc") is not None:
        return False
    return True

### using AdamW and Early stopping method

def get_seed_time(batch, seed_count, global_time_tensor =None):
    if hasattr(batch["transaction"], "time") and batch["transaction"].time is not None:
        return batch["transaction"].time[:seed_count].view(-1, 1).to(DEVICE)
    
    if global_time_tensor is not None:
        n_id = batch["transaction"].n_id[:seed_count]
        return global_time_tensor[n_id].view(-1,1).to(DEVICE)
    
    raise ValueError("No transaction time found in batch")




def train_experiment(name, comps, hg: DualHypergraph, hetero_train: HeteroData, train_idx: torch.Tensor,
                      hetero_val: HeteroData, val_idx: torch.Tensor):
    print(f"\n==== TRAIN: {name} =====")

    history =[]
    params = []
    for m in comps.values():
        if isinstance(m, nn.Module):
            params += list(m.parameters())

    if len(params) == 0:
        raise ValueError(f"No trainable parameters found for this experiment: {name}")
    
    opt = torch.optim.AdamW(params, lr= LR, weight_decay=WEIGHT_DECAY)  ### L2 Regularization /weight decay . which Reduces overfitting by 
    #discouraging overly large weights.

    if POS_WEIGHT is not None:
        loss_fn = nn.BCEWithLogitsLoss(pos_weight= POS_WEIGHT)
    else:
        loss_fn = nn.BCEWithLogitsLoss()

    scaler = torch.cuda.amp.GradScaler(enabled=(amp_ok_for_comps(comps)))
    train_loader = make_loader(hetero_train, train_idx, shuffle=True)

    if train_loader is None:
        raise ValueError(f"Empty training loader: {name}")  

    print("I m inside train function")
    
    avg_loss1 = 0.0
    best_val_auc = -1.0
    patience_left = EARLY_STOPPING_PATIENCE
    best_state = None


    for epoch in range(1, EPOCHS + 1):
        set_mode(comps, True)
        total_loss = 0.0
        n_processed = 0
        all_probs, all_labels = [], []

        for batch in train_loader:
            batch = batch.to(DEVICE)
            seed_count = batch["transaction"].batch_size
            if seed_count == 0:
                continue
            seed_nodes = batch["transaction"].n_id[:seed_count]
            y = batch["transaction"].y[:seed_count].float().to(DEVICE)

            opt.zero_grad(set_to_none= True)
            #print("0.0.0.0")

            with torch.cuda.amp.autocast(enabled = amp_ok_for_comps(comps)):
                if getattr(comps.get("fusion", None), "aux", False):
                    logits, aux_hgt, aux_hyper, trust = forward_variant(comps, hg, batch, seed_nodes, seed_count, return_aux=True)
                    loss_main = loss_fn(logits.float(), y.float())
                    loss_aux_hgt = loss_fn(aux_hgt.float(), y.float())
                    loss_aux_hyper = loss_fn(aux_hyper.float(), y.float())
                    loss = loss_main + 0.2 * loss_aux_hgt + 0.2 * loss_aux_hyper
                else:
                    logits = forward_variant(comps, hg, batch, seed_nodes, seed_count, return_aux = False)
                    loss = loss_fn(logits.float(), y.float())

            if not torch.isfinite(loss):
                print("|| Non-finite loss detected. Skipping batch. loss=", float(loss.detach().cpu()))
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)  #### AdamW only updates parameters when this is called here.
            scaler.update()

            #print("0.0.1.1")

            total_loss += float(loss.item()) * seed_count
            n_processed = n_processed + int(seed_count)
            probs = torch.sigmoid(logits.detach().float()).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y.detach().cpu().numpy())
        
        #print("2.check")
        if len(all_labels) == 0 or n_processed == 0 :
            print("No valid training batches this epoch")
            continue

        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)
        train_met = compute_metrics_updated(all_labels, all_probs, threshold=THRESH)

        print("Debug my code", type(train_met))

        val_met, _, _ = score_experiment(name, comps, hg, hetero_val, val_idx)
        avg_loss = total_loss / max(1, n_processed)

        train_auc = train_met["auc"]
        val_auc = val_met["auc"]

        avg_loss1 = avg_loss
        #if isinstance(met , tuple):
            #met = met[0]
             
        print(f"Epoch {epoch:02d} | loss={avg_loss:.5f} | train_auc = {train_auc:.4f} | val_auc = {val_auc:.4f}")

        improved = (val_auc > best_val_auc + EARLY_STOPPING_MIN_DELTA)
        if improved:
            best_val_auc = val_auc
            patience_left = EARLY_STOPPING_PATIENCE

            #### Saving best weght for each module

            best_state = {}
            for k, m in comps.items():
                if isinstance(m, nn.Module):
                    best_state[k] = {kk: vv.detach().cpu().clone() for kk, vv in m.state_dict().items()}

        else:
            patience_left = patience_left - 1
            if patience_left <= 0:
                print(f"[Early Stopping] stop at epoch ={epoch}, best_val_auc ={best_val_auc}")
                break

    if best_state is not None:
        for k, sd in best_state.items():
            if isinstance(comps.get(k), nn.Module):
                comps[k].load_state_dict(sd) 

    return best_val_auc               



@torch.no_grad()
def eval_experiment(name, comps, hg: DualHypergraph, hetero_full: HeteroData, test_idx: torch.Tensor):
    print(f"\n====EVAL: {name}=====")
    set_mode(comps, False)
    loader = make_loader(hetero_full, test_idx, shuffle=False)

    print("eval_experiment....")

    all_probs, all_labels = [], []
    for batch in loader:
        batch = batch.to(DEVICE)
        seed_count = batch["transaction"].batch_size
        seed_nodes = batch["transaction"].n_id[:seed_count]
        y = batch["transaction"].y[:seed_count].float()

        with torch.cuda.amp.autocast(enabled= False):
            logits = forward_variant(comps, hg, batch, seed_nodes, seed_count)
            probs = torch.sigmoid(logits.float())

        #probs = forward_variant(comps, hg, batch, seed_nodes, seed_count)
        all_probs.append(probs.detach().float().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels)    

    met = compute_metrics_updated(y_true, y_prob, threshold=THRESH)
    print(f"acc={met['acc']:.4f} prec={met['prec']:.4f} rec={met['rec']:.4f} f1={met['f1']:.4f} auc={met['auc']:.4f}")
    return met, y_true, y_prob



###XGBoost###
def run_xgboost_baseline(df: pd.DataFrame,
                         train_idx: torch.Tensor,
                         test_idx: torch.Tensor,
                         tdt_min: float,
                         tdt_max: float,
                         threshold: float = THRESH):
    if xgb is None:
        print("XGBoost not installed.")
        return None
    print("[XGBoost] ===TRAIN+EVAL ====")
    amt = df["TransactionAmt"].values.astype(np.float32)
    tdt = df["TransactionDT"].values.astype(np.float32)
    tdt_norm = (tdt - tdt_min) / (tdt_max - tdt_min + 1e-6)

    delta_t = df["delta_t"].values.astype(np.float32)
    txn_count_1h = df["txn_count_1h"].values.astype(np.float32)
    txn_count_24h = df["txn_count_24h"].values.astype(np.float32)
    amt_zscore = df["amt_zscore"].values.astype(np.float32)

    X = np.column_stack([amt, tdt_norm, delta_t,txn_count_1h, txn_count_24h, amt_zscore ])

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    y = df["isFraud"].values.astype(np.int32)
    tr = train_idx.cpu().numpy()
    te = test_idx.cpu().numpy()
    X_train , y_train = X[tr] , y[tr]
    X_test, y_test = X[te] , y[te]

    pos = max(1, int((y_train == 1).sum()))
    neg = max(1, int((y_train == 0).sum()))
    scale_pos_weight = neg / pos

    clf = xgb.XGBClassifier(
        n_estimators =800,
        max_depth = 6,
        learning_rate = 0.05,
        subsample = 0.8,
        colsample_bytree = 0.8,
        reg_lambda = 1.0,
        min_child_weight = 1,
        objective = "binary:logistic",
        eval_metric ="auc",
        tree_method ="hist",
        n_jobs =max(1, (os.cpu_count()  or 4) // 2),
        scale_pos_weight = scale_pos_weight,
        random_state = SEED
    )


    clf.fit(X_train , y_train)
    y_prob = clf.predict_proba(X_test)[:, 1].astype(np.float32)

    met = compute_metrics_updated(y_test, y_prob , threshold= threshold)
    print(f"[XGBoost]  acc={met['acc']:.4f}  prec={met['prec']:.4f} rec={met['rec']:.4f}"
          f"f1={met['f1']:.4f}  auc={met['auc']:.4f}")
    
    plot_confusion_matrix(y_test , y_prob, threshold=threshold, title="CM: XGBoost(tabular baseline)")
    return met


class FastRingExtractorCausal:
    def __init__(self, data: HeteroData, labels: np.ndarray, txn_time: np.ndarray):
        self.data = data
        self.labels = labels.astype(int)
        self.txn_time = txn_time.astype(np.float64)

        self.txn_to_attr = {}
        self.attr_to_txn = {}

        for (src, rel, dst) in data.edge_types:
            ei = data[(src, rel, dst)].edge_index
            if ei is None or ei.numel() == 0:
                continue

            if src == "transaction":
                self.txn_to_attr.setdefault(dst, {})
                t_list = ei[0].tolist()
                a_list = ei[1].tolist()
                for t, a in zip(t_list, a_list):
                    self.txn_to_attr[dst].setdefault(t, []).append(a)

            if dst == "transaction":
                self.attr_to_txn.setdefault(src, {})
                a_list = ei[0].tolist()
                t_list = ei[1].tolist()
                for a, t in zip(a_list, t_list):
                    self.attr_to_txn[src].setdefault(a, []).append(t)


    def extract(self, txn_id: int, hops: int =2, causal: bool = True ) -> nx.Graph:
        num_txn = self.data["transaction"].num_nodes
        if txn_id < 0 or txn_id >= num_txn:
            raise IndexError(f"txn_id {txn_id} out of bound [0, {num_txn-1}]")
        
        seed_time = self.txn_time[txn_id]

        G = nx.Graph()
        G.add_node(f"txn_{txn_id}", type = "txn", fraud = int(self.labels[txn_id]), time = float(seed_time))
        frontier = {txn_id}
        visited_txn = {txn_id}
        visited_attr = set()

        for _ in range(hops):
            next_frontier = set()
            for t in list(frontier):
                for attr_type, mapping in self.txn_to_attr.items():
                    if t not in mapping:
                        continue
                    for attr_id in mapping[t]:
                        attr_node = f"{attr_type}_{attr_id}"
                        if attr_node not in visited_attr:
                            visited_attr.add(attr_node)
                            G.add_node(attr_node, type=attr_type)
                        G.add_edge(f"txn_{t}", attr_node)

                        for t2 in self.attr_to_txn.get(attr_type, {}).get(attr_id, []):
                            if causal and self.txn_time[t2] > seed_time:
                                continue
                            if t2 not in visited_txn:
                                visited_txn.add(t2)
                                next_frontier.add(t2)
                                G.add_node(f"txn_{t2}", type ="txn", fraud = int(self.labels[t2]), time = float(self.txn_time[t2]))
                            G.add_edge(attr_node, f"txn_{t2}")
            frontier = next_frontier
        return G                        


def ring_consistency_metrics(G: nx.Graph):
    txn_nodes = [n for n in G.nodes if G.nodes[n].get("type") == "txn"]
    attr_nodes = [n for n in G.nodes if G.nodes[n].get("type") != "txn"]
    n_txn = len(txn_nodes)
    if n_txn < 2 or len(attr_nodes) == 0:
        return{
            "pair_multi_attr_frac": 0.0,
            "node_multi_attr_frac": 0.0,
            "avg_shared_attr_types": 0.0,
            "ring_consistency_score": 0.0,
        }
    
    attr_to_neighbors = []
    for a in attr_nodes:
        a_type = G.nodes[a].get("type", "unknown")
        neigh = [u for u in G.neighbors(a) if G.nodes[u].get("type") == "txn"]
        if len(neigh) >= 2:
            attr_to_neighbors.append((a_type, sorted(neigh)))

    pair_shared_types = defaultdict(set)
    for a_type, neigh in attr_to_neighbors:
        for i in range(len(neigh)):
            for j in range(i+1, len(neigh)):
                pair_shared_types[(neigh[i], neigh[j])].add(a_type)

    all_pair_counts = []
    for i in range(n_txn):
        for j in range(i +1 , n_txn):
            t1, t2 = txn_nodes[i], txn_nodes[j]
            key =(t1,t2) if (t1, t2) in pair_shared_types else (t2, t1)
            all_pair_counts.append(len(pair_shared_types.get(key, set())))

    pair_multi_attr_frac = float(np.mean([c >= 2 for c in all_pair_counts])) if all_pair_counts else 0.0

    txn_has_multi = {t: False for t in txn_nodes}
    for(t1, t2), typeset in pair_shared_types.items():
        if len(typeset) >=2:
            txn_has_multi[t1] = True
            txn_has_multi[t2] = True

    node_multi_attr_frac = float(np.mean(list(txn_has_multi.values()))) if txn_nodes else 0.0

    connected_counts = [c for c in all_pair_counts if c > 0]
    avg_shared_attr_types = float(np.mean(connected_counts)) if connected_counts else 0.0

    return{
        "pair_multi_attr_frac": pair_multi_attr_frac,
        "node_multi_attr_frac": node_multi_attr_frac,
        "avg_shared_attr_types": avg_shared_attr_types,
        "ring_consistency_score": 0.5 * (pair_multi_attr_frac + node_multi_attr_frac),

    }

##### RING Logic -- my new addtion
def enhancedRingMetrics(G: nx.Graph, txn_id: int):
    txn_nodes = [n for n in G.nodes if G.nodes[n].get("type") == "txn"]
    attr_nodes = [n for n in G.nodes if G.nodes[n].get("type") != "txn"]

    n_txn = len(txn_nodes)
    n_attr = len(attr_nodes)
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()

    density = nx.density(G) if n_nodes > 1 else 0.0
    density = float(np.clip(density, 0.0, 1.0))

    fraud_flags = [int(G.nodes[n].get("fraud", 0)) for n in txn_nodes]
    fraud_rate = float(np.mean(fraud_flags)) if len(fraud_flags) > 0 else 0.0
    if math.isnan(fraud_rate):
        fraud_rate = 0.0 

    pair_total = 0
    pair_any = 0
    pair_multi = 0

    for i, t1 in enumerate(txn_nodes):
        n1 = set(G.neighbors(t1))
        for t2 in txn_nodes[i +1:]:
            n2 = set(G.neighbors(t2))
            shared = n1 & n2
            pair_total = pair_total + 1
            if len(shared) > 0:
                pair_any = pair_any + 1
            if len(shared) >= 2:
                pair_multi = pair_multi + 1

    pair_connacted_frac = pair_any / max(pair_total, 1)
    pair_multi_attr_frac = pair_multi / max(pair_total, 1) 

    seed = f"txn_{txn_id}"
    if seed in G.nodes and n_txn > 1:
        seed_neighbors = set(G.neighbors(seed))
        seed_multi_links = 0
        for t in txn_nodes:
            if t == seed:
                continue
            if len(seed_neighbors & set(G.neighbors(t))) >=2:
                seed_multi_links = seed_multi_links + 1
        node_multi_attr_frac = seed_multi_links / max(n_txn -1, 1) 
    else:
        node_multi_attr_frac = 0.0

    attr_types = [G.nodes[n].get("type", "unknown") for n in attr_nodes]
    if len(attr_types) == 0:
        attr_type_entropy = 0.0
    else:
        _, cnt = np.unique(attr_types, return_counts= True) 
        attr_type_entropy = _shannon_entropy_from_counts(cnt) 

    size_norm = min(n_txn /50.0, 1.0) 

    ring_consistency_score = float(np.clip(
        0.20 * size_norm
        + 0.20 * density
        + 0.20 * fraud_rate
        + 0.20 * pair_multi_attr_frac + 0.20 * node_multi_attr_frac, 0.0, 1.0
    ))  

    return {
        "ring_txn_count": int(n_txn),
        "ring_attr_count": int(n_attr),
        "ring_size": int(n_nodes),
        "ring_edge_count": int(n_edges),
        "ring_fraud_rate": float(fraud_rate),
        "ring_density": float(density),
        "pair_connected_frac": float(pair_connacted_frac),
        "pair_multi_attr_frac": float(pair_multi_attr_frac),
        "node_multi_attr_frac": float(node_multi_attr_frac),
        "attr_type_entropy": float(attr_type_entropy),
        "ring_consistency_score": float(ring_consistency_score),
    }   


@dataclass
class BudgetResult:
    K: int
    alpha_star : float
    precision_fixed_alpha: float
    precision_learned_alpha: float
    precision_model_only: float
    precision_ring_only: float
    precision_gain: float

class AdaptiveBudgetOptimizer:
    def __init__(self, budgets=(25, 50, 100, 200), fixed_alpha=0.7, n_alpha_grid=101):
        self.budgets = tuple(int(x) for x in budgets)
        self.fixed_alpha = float(fixed_alpha)
        self.alpha_grid = np.linspace(0.0, 1.0, int(n_alpha_grid))

    @staticmethod
    def _minmax(x):
        x = np.asarray(x, dtype=np.float64) 
        lo, hi = x.min(), x.max()
        if hi - lo < 1e-12:
            return np.zeros_like(x)
        return (x - lo) / (hi - lo) 
    
    @staticmethod
    def _combine(m, r, a):
        return a * m + (1.0 - a) * r

    @staticmethod
    def _precision_at_k(labels, scores, K):
        labels = np.asarray(labels, dtype=np.int64)
        scores = np.asarray(scores, dtype=np.float64)
        K = min(int(K), len(labels))
        if K <= 0:
            return 0.0
        top = np.argsort(scores)[::-1][:K]
        return float(labels[top].sum() / K)
    
    @staticmethod
    def _recall_at_k(labels, scores, K):
        labels = np.asarray(labels, dtype=np.int64)
        scores = np.asarray(scores, dtype=np.float64)
        K = min(int(K), len(labels))
        total_pos = int(labels.sum())
        if K <= 0 or total_pos <=0:
            return 0.0
        top = np.argsort(scores)[::-1][:K]
        return float(labels[top].sum() / total_pos)
    
    def fit(self, model_scores, ring_scores, labels):
        model_scores = self._minmax(model_scores)
        ring_scores = self._minmax(ring_scores)
        labels = np.asarray(labels).astype(int)

        out = {}
        alpha_list = []

        for K in self.budgets:
            curve = []
            for a in self.alpha_grid:
                s = self._combine(model_scores, ring_scores, a)
                curve.append(self._precision_at_k(labels, s, K))
            curve = np.asarray(curve, dtype = np.float64) 

            idx = int(np.argmax(curve)) 
            alpha_star = float(self.alpha_grid[idx]) 

            p_fixed = self._precision_at_k(labels, self._combine(model_scores, ring_scores, self.fixed_alpha), K)
            p_learn = self._precision_at_k(labels, self._combine(model_scores, ring_scores, alpha_star), K)
            p_model = self._precision_at_k(labels, model_scores, K)
            p_ring = self._precision_at_k(labels, ring_scores, K)

            out[K] = BudgetResult(
                K = K,
                alpha_star=alpha_star,
                precision_fixed_alpha=float(p_fixed),
                precision_learned_alpha=float(p_learn),
                precision_model_only=float(p_model),
                precision_ring_only = float(p_ring),
                precision_gain=float(p_learn - p_fixed),
            )
            alpha_list.append(alpha_star)

        rho, pval = (None, None)
        if spearmanr is not None and len(alpha_list) >= 3:
            budgets_used = np.array(self.budgets[:len(alpha_list)],dtype = np.float64)
            rr, pp = spearmanr(budgets_used, np.array(alpha_list, dtype=np.float64))
            rho = float(rr) if rr == rr else None
            pval = float(pp) if pp == pp else None

        summary = pd.DataFrame([{
            "budget_K": k,
            "alpha_star": v.alpha_star,
            "P@K_fixed": v.precision_fixed_alpha,
            "P@K_learned": v.precision_learned_alpha,
            "P@K_model_only": v.precision_model_only,
            "P@K_ring_only": v.precision_ring_only,
            "gain": v.precision_gain,
            "R@K_fixed": self._recall_at_k(labels, self._combine(model_scores, ring_scores, self.fixed_alpha), k),
            "R@K_learned": self._recall_at_k(labels, self._combine(model_scores, ring_scores, v.alpha_star), k),
        } for k, v in out.items()]).sort_values("budget_K").reset_index(drop=True)

        return{
            "results": out,
            "summary_df": summary,
            "spearman_rho": rho,
            "spearman_pval": pval,
        }    



def compute_ring_metrics_row(G: nx.Graph, txn_id: int, prob: float, label: int, pred: int, hops: int):
    nodes = list(G.nodes())
    edges = list(G.edges())

    txn_nodes = [n for n in nodes if G.nodes[n].get("type") == "txn"]
    attr_nodes = [n for n in nodes if G.nodes[n].get("type")!= "txn"]

    n_nodes = len(nodes)
    n_edges = len(edges)
    density = nx.density(G) if n_nodes > 1 else 0.0

    degs = np.array([d for _, d in G.degree()], dtype = np.int64) if n_nodes > 0 else np.array([0])
    avg_degree = float(degs.mean()) if len(degs) else 0.0
    max_degree = int(degs.max()) if len(degs) else 0

    fraud_flags = [int(G.nodes[n].get("fraud",0)) for n in txn_nodes]
    fraud_txn_frac = float(np.mean(fraud_flags)) if len(fraud_flags) else 0.0

    attr_types = [G.nodes[n].get("type", "unknown") for n in attr_nodes]
    if len(attr_nodes) ==0:
        attr_type_entropy = 0.0
    else:
        _, cnt = np.unique(attr_types, return_counts= True)
        attr_type_entropy = _shannon_entropy_from_counts(cnt)  

    rcs = ring_consistency_metrics(G)
    novel = enhancedRingMetrics(G, txn_id=txn_id)

    group = "TP" if (pred ==1 and label == 1) else ("FP" if (pred == 1 and label == 0) else "OTHER")  

    row = {
        "txn_id": int(txn_id),
        "hops": int(hops),
        "prob": float(prob),
        "label": int(label),
        "pred": int(pred),
        "group": group,
        "n_nodes": int(n_nodes),
        "n_edges": int(n_edges),
        "density": float(density),
        "avg_degree": float(avg_degree),
        "max_degree": int(max_degree),
        "fraud_txn_frac": float(fraud_txn_frac),
        "attr_type_entropy": float(attr_type_entropy),
    } 
    row.update(rcs)
    row.update(novel)
    row["ring_consistency_score"] = row["ring_consistency_score"]
    return row

@torch.no_grad()
def score_full_fusion_on_test(
    hg: DualHypergraph,
    hetero_full: HeteroData,
    test_idx: torch.Tensor,
    hyp: nn.Module,
    het: nn.Module,
    fusion: nn.Module,
):
    hyp.eval(); het.eval(); fusion.eval()
    loader = NeighborLoader(
        hetero_full,
        input_nodes=("transaction", test_idx),
        num_neighbors=list(NEIGHBORS),
        batch_size = BATCH_SIZE,
        shuffle = False

    )

    probs_list = []
    ids_list = []
    trustList = []

    for batch in loader:
        batch = batch.to(DEVICE)
        seed_count = batch["transaction"].batch_size
        seed_nodes = batch["transaction"].n_id[:seed_count]

        he_ids = hg.get_batch_hyperedges(seed_nodes, device=DEVICE)
        hyper_emb = hyp(seed_nodes, he_ids)
        tx_emb_all = het(batch)
        hetero_emb = tx_emb_all[:seed_count]

        ### here  FusiontestFinal return the value trust
        if getattr(fusion, "aux", False):
            logits,_ ,_ , trust = fusion(hyper_emb,hetero_emb, return_aux = True)
            trustList.append(trust.detach().cpu().numpy())
        else:
            logits = fusion(hyper_emb, hetero_emb)

        p = torch.sigmoid(logits.float()).detach().cpu().numpy()
        probs_list.append(p)
        ids_list.append(seed_nodes.detach().cpu().numpy())

    probs_np = np.concatenate(probs_list)
    test_ids_np = np.concatenate(ids_list)
    trustListNp = np.concatenate(trustList) if len(trustList) else None
    return test_ids_np, probs_np, trustListNp

def run_offline_ring_stage(
    test_ids_np: np.ndarray,
    probs_np: np.ndarray,
    labels_global: np.ndarray,
    txn_time_global: np.ndarray,
    hetero_full: HeteroData ,
    trustNp: np.ndarray = None  
):
    flagged_mask = probs_np >= RINGTHRESH
    flagged_ids = test_ids_np[flagged_mask]
    flagged_probs = probs_np[flagged_mask]

    flagTrust = None
    if trustNp is not None:
        trustNp = np.asarray(trustNp, dtype=np.float32).reshape(-1)
        if len(trustNp) != len(probs_np):
            print("Ignorning Trust here")
            flagTrust = None
        else:
            flagTrust = trustNp[flagged_mask]  

    print(f"[RingStage] test_total ={len(test_ids_np):,} test_flagged={len(flagged_ids):,} threshold ={THRESH}")

    if len(flagged_ids) == 0:
        print("[RingStage]No flagged test transactions at this threshold")
        return None, None, None, None
    
    #Keep top k flagged
    if RING_TOPK_FLAGGED is not None and len(flagged_ids) > RING_TOPK_FLAGGED:
        order = np.argsort(-flagged_probs)[:RING_TOPK_FLAGGED]
        flagged_ids = flagged_ids[order]
        flagged_probs = flagged_probs[order]
        if flagTrust is not None:
            flagTrust = flagTrust[order]
        print(f"[ringStage] Using top_k flagged = {len(flagged_ids):,}")

    extractor = FastRingExtractorCausal(
        data = hetero_full,
        labels = labels_global,
        txn_time= txn_time_global
    ) 

    rows = []
    for i, (txn_id, prob) in  enumerate(zip(flagged_ids, flagged_probs)):
        txn_id = int(txn_id)
        G = extractor.extract(txn_id, hops = RING_HOPS, causal = RING_CAUSAL)
        label = int(labels_global[txn_id])
        row = compute_ring_metrics_row(
            G,
            txn_id=txn_id,
            prob=float(prob),
            label= label, pred= 1, hops= RING_HOPS
        )

        if flagTrust is not None:
            row["trust"] =float(flagTrust[i])
        else:
            row["trust"] = float("nan") 

        if row["group"] in ("TP", "FP"):
            rows.append(row)

    dfm = pd.DataFrame(rows)
    if dfm.empty:
        print("[RingStage]  No TP/FP found in selected flagged test set.")
        return dfm, None, None , None

    print("\n[RingScience] TP/FP counts:")
    print(dfm["group"].value_counts())

    ##optional statistical tests
    if mannwhitneyu is not None:
        print("\n[RingScience] Mann Whitney U tests(TP vs FP): ")
        for metric in ["ring_consistency_score", "ring_density", "attr_type_entropy","ring_fraud_rate","pair_multi_attr_frac"]:
            tp = dfm.loc[dfm.group == "TP", metric].values
            fp = dfm.loc[dfm.group == "FP", metric].values
            if len(tp) < 5 or len(fp) < 5:
                print(f"  {metric}: not enough samples")
                continue
            stat, p = mannwhitneyu(tp, fp, alternative= "two-sided")
            print(f" {metric}: U={stat:.1f}, p={p:.4g}")

    queue_df, comp_df = build_investigation_queue(dfm, budgets= RING_BUDGETS)  

    adaptive_df = None
    if len(dfm) > 0 and int(dfm["label"].sum()) > 0:
        abo = AdaptiveBudgetOptimizer(
            budgets= RING_BUDGETS,
            fixed_alpha= RING_ALPHA,
            n_alpha_grid= 101
        )
        fit_res = abo.fit(
            model_scores= dfm["prob"].values,
            ring_scores= dfm["ring_consistency_score"].values,
            labels= dfm["label"].values
        )
        adaptive_df = fit_res["summary_df"]
        print("My Ring logic and Adaptive Budget summary")
        print(adaptive_df.to_string(index=False))
        print(f"[Ring Logic] Spearman rho(K, alpha*)={fit_res['spearman_rho']}, p={fit_res['spearman_pval']}")


    if "trust" in dfm.columns and dfm["trust"].notna().any():
        dfm["Score-CombinedTrust"] = combineMytrustscore(
            prob=dfm["prob"].values,
            ringscore=dfm["ring_consistency_score"].values,
            trust = dfm["trust"].values,
            alp0=RING_ALPHA,
            gama=0.5
        )
    if queue_df is not None:
        print("\n[RingStage] Investigation queue built")
        print(queue_df.head(10).to_string(index=False))    

    if comp_df is not None:
        print("\n[RingStage] Budget comparison")
        print(comp_df.to_string(index=False))  

    return dfm, queue_df, comp_df, adaptive_df

 


############################
## Heterogeneous GNN (HGTConv)
############################
class HeteroHGTModel(nn.Module):
    def __init__(self, metadata, node_sizes,tx_feat_dim, hidden = HIDDEN_DIM,time_dim=TIME_DIM, heads =4, use_time = True, dropout_p = 0.2):
        super().__init__()
        node_types, _ = metadata
        #self.hidden = hidden
        #self.debug = debug
        self.use_time = use_time
        self.dropout = nn.Dropout(dropout_p)

        self.enc = nn.ModuleDict()
        for nt in node_types:
            if nt == "transaction":
                in_dim = tx_feat_dim + (time_dim if use_time else 0)
                self.enc[nt] = nn.Linear(in_dim,  hidden)
            else:
                self.enc[nt] = nn.Embedding(int(node_sizes[nt]), hidden)
        ## in_channels = the dimension of input node embeddings to the HGT layer
        ## out_channels = the dimension of output node embeddings produced by the layer
        if use_time:
            self.time2vec = Time2Vector(out_dim = TIME_DIM)  ## encode raw time
            self.temp_att = TemporalAttention( time_dim = TIME_DIM)
        else:
            self.time2vec = None
            self.temp_att = None

        self.hgt1 = HGTConv(in_channels = hidden, out_channels = hidden, metadata = metadata, heads = heads)
        self.hgt2 = HGTConv(in_channels= hidden,out_channels = hidden, metadata = metadata, heads = heads )

        self.ln = nn.ModuleDict({nt: nn.LayerNorm(hidden) for nt in node_types})
        self.transaction_time = None
        
    def _get_node_ids(self, batch: HeteroData, ntype: str):
        if hasattr(batch[ntype], "n_id") and batch[ntype].n_id is not None:
            ids = batch[ntype].n_id.long()
        elif hasattr(batch[ntype], "x") and batch[ntype].x is not None: 
            raw = batch[ntype].x
            if raw.dim() ==1:
                ids = raw.view(-1).long()
            else:
                ids = raw[:, 0].long()    

        else:
            ids = torch.arange(batch[ntype].num_nodes, device = DEVICE, dtype = torch.long)

        max_id = self.enc[ntype].num_embeddings - 1
        return ids.clamp(0, max_id)    


    def forward(self, batch: HeteroData):
        x_dict = {}
        for ntype in batch.node_types:
            if ntype == "transaction":
                tx_feats = batch["transaction"].x.float()

                if self.use_time:
                    if hasattr(batch["transaction"], "time") and batch["transaction"].time is not None:
                        t_raw = batch["transaction"].time.view(-1, 1).to(tx_feats.device)
                    else:
                        if self.transaction_time is None:
                            raise ValueError("Set hetero_model.transaction time before training/inference.")
                        n_id = batch["transaction"].n_id
                        t_raw = self.transaction_time[n_id].view(-1, 1).to(tx_feats.device)

                    t_emb = self.time2vec(t_raw)
                    t_att, _ = self.temp_att(t_emb)
                    tx_input = torch.cat([ tx_feats, t_att], dim =1) ### concatenate features with time embedding
                #x_dict[ntype]  = tx_input
                #x_dict[ntype] = self.enc["transaction"](tx_input)
                else:
                    tx_input = tx_feats
                 
                x_dict["transaction"] = self.enc["transaction"](tx_input)
                 #x_dict[ntype] = x.long()
                #x_dict[ntype] = self.enc[ntype](x.long())
            else:
                ids = self._get_node_ids(batch, ntype)
                x_dict[ntype] = self.enc[ntype](ids.long())

               
        edge_index_dict = { rel: ei for rel, ei in batch.edge_index_dict.items() if ei is not None and ei.numel() > 0  }
        if len(edge_index_dict) == 0:
            return x_dict["transaction"] 
        #if not edge_index_dict:
            #return x_dict.get("transaction", torch.empty(0, self.type_enc["transaction"].out_features, device = DEVICE))

        h1 = self.hgt1(x_dict, edge_index_dict)
        for k in list(h1.keys()):
            h1[k] = self.dropout(F.relu(self.ln[k](h1[k])))

        h2 = self.hgt2(h1, edge_index_dict)
        for k in list(h2.keys()):
            h2[k] = self.dropout(F.relu(self.ln[k](h2[k])))

        return h2["transaction"]    
    

                
class BinaryHead(nn.Module):
    def __init__(self, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, emb):
        return self.fc(emb).squeeze(1)   

                   
##################
### Fusion Model (HyperGraph + HGT)+ MLP
###################

class FusionGated(nn.Module):
    def __init__(self, hidden_dim = HIDDEN_DIM, dropout_p = 0.1):
        super().__init__()
        self.ln_a = nn.LayerNorm(hidden_dim)
        self.ln_b = nn.LayerNorm(hidden_dim)

        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim , hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, a_emb, b_emb):
        a = self.ln_a(a_emb)
        b = self.ln_b(b_emb)
        alpha = torch.sigmoid(self.gate(torch.cat([a, b], dim = 1)))
        fused = alpha * a + (1.0 - alpha) * b
        return self.head(fused).squeeze(1)


class Fusion(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, hyper_emb, hetero_emb):
        return self.fc(torch.cat([hyper_emb, hetero_emb], dim = 1)).squeeze(1) 
    
class FusiontestFinal(nn.Module):
    def __init__(self, hidden_dim = HIDDEN_DIM, dropout_hyper = 0.35, dropout_gate = 0.20,
                 n_heads =4,
                 aux = True, 
                 gate_detach_hgt = False):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.aux = aux
        self.gate_detach_hgt = gate_detach_hgt

        self.In_hgt = nn.LayerNorm(hidden_dim)
        self.In_hyper = nn.LayerNorm(hidden_dim)

        self.hyper_dropout = nn.Dropout(dropout_hyper)
        self.gate_dropout = nn.Dropout(dropout_gate)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=n_heads,
            dropout=dropout_hyper,
            batch_first=True
        )

        self.trust_gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim //2 ),
            nn.ReLU(),
            nn.Dropout(dropout_gate),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

        self._beta_logit = nn.Parameter(torch.tensor(-4.0))

        self.post = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_gate),
            nn.Linear(hidden_dim, hidden_dim),
        )
        nn.init.zeros_(self.post[-1].weight)
        nn.init.zeros_(self.post[-1].bias)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout_gate),
            nn.Linear(64, 1),
        )

        if aux:
            self.aux_hgt_head = nn.Linear(hidden_dim, 1)
            self.aux_hyper_head = nn.Linear(hidden_dim, 1)

    def beta(self):
        return torch.sigmoid(self._beta_logit)

    def forward(self, hyper_emb, hetero_emb, return_aux = False):
        hgt_base = self.In_hgt(hetero_emb)
        hyper = self.In_hyper(hyper_emb)
        hyper = self.hyper_dropout(hyper)

        q = hgt_base.unsqueeze(1)
        kv = hyper.unsqueeze(1)
        attn_out, _ = self.cross_attn(query=q, key=kv, value=kv)
        corr_vec = attn_out.squeeze(1)
        corr_vec = self.hyper_dropout(corr_vec)
        gate_hgt = hgt_base.detach() if self.gate_detach_hgt else hgt_base
        trust = self.trust_gate(torch.cat([gate_hgt, hyper], dim = -1))
        trust = self.gate_dropout(trust)

        scale = self.beta()
        correction = (scale * trust) * corr_vec
        fused = hgt_base + correction
        fused = fused + self.post(fused)

        logits = self.classifier(fused).squeeze(-1)
        if return_aux:
            if not self.aux:
                return logits, None, None, trust.squeeze(-1)
            aux_hgt = self.aux_hgt_head(hgt_base).squeeze(-1)
            aux_hyper = self.aux_hyper_head(hyper).squeeze(-1)
            return logits, aux_hgt, aux_hyper, trust.squeeze(-1)
        
        return logits 

class TxnMLP(nn.Module):
    def __init__(self, in_dim, hidden = HIDDEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1) 


class Time2VecTxnEncoder(nn.Module):
    #### this is for Fusion(Hyper + Time2Vec)   
    def __init__(self, tx_feat_dim, hidden = HIDDEN_DIM, time_dim = TIME_DIM, dropout_p = 0.1):
        super().__init__()
        self.time2vec = Time2Vector(out_dim= time_dim)
        self.temp_att = TemporalAttention(time_dim= time_dim)
        self.proj = nn.Linear(tx_feat_dim + time_dim, hidden)
        self.ln = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, tx_feats: torch.Tensor, t_raw: torch.Tensor):
        t_emb = self.time2vec(t_raw) 
        t_att, _ =self.temp_att(t_emb)  
        x = torch.cat([tx_feats, t_att], dim =1)
        h = self.dropout(F.relu(self.ln(self.proj(x))))
        return h
     

@torch.no_grad()
def score_experiment(name, comps, hg: DualHypergraph, hetero_data: HeteroData, idx: torch.Tensor):
    print(f"\n====EVALUATION: {name}=====")
    set_mode(comps, False)
    loader = make_loader(hetero_data, idx, shuffle=False)

    if loader is None:
        return compute_metrics_updated([],[]), np.array([]), np.array([])


    all_probs, all_labels = [], []
    for batch in loader:
        batch = batch.to(DEVICE)
        seed_count = batch["transaction"].batch_size
        if seed_count == 0:
            continue
        seed_nodes = batch["transaction"].n_id[:seed_count]
        y = batch["transaction"].y[:seed_count].float()

        with torch.cuda.amp.autocast(enabled= False):
            logits = forward_variant(comps, hg, batch, seed_nodes, seed_count)
            probs = torch.sigmoid(logits.float())

        #probs = forward_variant(comps, hg, batch, seed_nodes, seed_count)
        all_probs.append(probs.detach().float().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels)    

    met = compute_metrics_updated(y_true, y_prob, threshold=THRESH)
    #print(f"acc={met['acc']:.4f} prec={met['prec']:.4f} rec={met['rec']:.4f} f1={met['f1']:.4f} auc={met['auc']:.4f}")
    return met, y_true, y_prob

   

#####################
#### Training
#####################

class TxnDataset(Dataset):
    def __init__(self, labels): self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return i, self.labels[i]


        
  
def evaluate_model(hyp_model, het_model, fusion_model, hypergraph, hetero_data, mask =None):

    hyp_model.eval()
    het_model.eval()
    fusion_model.eval()

    if mask is None:
        input_nodes = ("transaction", torch.arange(hypergraph.num_nodes))
    else:
        input_nodes = ("transaction", mask)

   
    loader = NeighborLoader(hetero_data, input_nodes = input_nodes, num_neighbors = NEIGHBORS, batch_size = BATCH_SIZE, shuffle = False)

       
    all_preds =[]
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            seed_count = batch["transaction"].batch_size
            seed_nodes = batch["transaction"].n_id[:seed_count]
            hyper_emb = hyp_model(hypergraph.H_row.to(DEVICE) , hypergraph.H_col.to(DEVICE) , seed_nodes)
            tx_emb_batch = het_model(batch)
            tx_emb_seeds = tx_emb_batch[:seed_count]
            
            ## Fusion prediction-> Fraud probabilities
            logits = fusion_model(hyper_emb, tx_emb_seeds)
            preds = torch.sigmoid(logits).detach().cpu().numpy()
            labels = batch["transaction"].y[:seed_count].detach().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels)

    all_preds_np = np.concatenate(all_preds)
    all_labels_np = np.concatenate(all_labels)
    return compute_metrics_updated(all_labels_np, all_preds_np, threshold=THRESH) , (all_labels_np, all_preds_np)
        
############3 Add rolling behavirol features per entity  example card, user
def add_temporal_behavior_features(df: pd.DataFrame, entity_col: str ="card1", exclude_current: bool = True) -> pd.DataFrame:
    #### Assume Data Frame is sorted by time.
    #df = df.copy()
    #df["_row_id__"] = np.arange(len(df), dtype =np.int64)
    print("card1 null %:", df["card1"].isna().mean())
    print("TransactionTime NaT %:", pd.to_datetime(df["TransactionDT"], unit="s", origin="unix", errors ="coerce").isna().mean())
    print("rows:", len(df))
    out = df.copy()
    out["_row_id__"] = np.arange(len(out), dtype =np.int64)

    out["TransactionTime"] = pd.to_datetime(out["TransactionDT"], unit="s", origin="unix", errors="coerce")
    out[entity_col] = out[entity_col].astype("object").where(out[entity_col].notna(), "__missing__")

    df_sorted = out.sort_values([entity_col, "TransactionTime", "_row_id__"], kind="mergesort").reset_index(drop=True)
    #Time delta since previos transaction . delta_t per entity(seconds)
    df_sorted["delta_t"] = (
        df_sorted.groupby(entity_col, sort =False, dropna = False )["TransactionTime"]
          .diff()
          .dt.total_seconds()
          .fillna(0.0)
          .clip(lower=0.0)
          .astype("float32")

    )

    ## Time based Rolling transaction counts
    ##Rolling feature computed inside each group

    df_sorted["txn_count_1h"] = 0.0
    df_sorted["txn_count_24h"] = 0.0
    df_sorted["amt_mean_24h"] = 0.0
    df_sorted["amt_std_24h"] = 1.0
    df_sorted["amt_zscore"] = 0.0
    
    valid_time = df_sorted["TransactionTime"].notna()
    if valid_time.any():
        dfx = df_sorted.loc[valid_time, [entity_col, "TransactionTime", "TransactionDT", "TransactionAmt"]].copy()

        g = dfx.groupby(entity_col, sort = False, dropna = False)

        c1 = (
            g.rolling("1h", on ="TransactionTime", min_periods = 1)["TransactionDT"]
            .count()
            .reset_index(level = 0, drop = True)
        )

        c24 = (
            g.rolling("24h", on ="TransactionTime", min_periods = 1)["TransactionDT"]
            .count()
            .reset_index(level = 0, drop = True)
        )

        m24 = (
            g.rolling("24h", on ="TransactionTime", min_periods = 1)["TransactionAmt"]
            .mean()
            .reset_index(level = 0, drop = True)
        )

        s24 = (
            g.rolling("24h", on ="TransactionTime", min_periods = 1)["TransactionAmt"]
            .std()
            .reset_index(level = 0, drop = True)
        )

        if exclude_current:
            keys = dfx[entity_col] 
            c1 = c1.groupby(keys , sort = False, dropna = False).shift(1).fillna(0.0) 
            c24 = c24.groupby(keys , sort = False, dropna = False).shift(1).fillna(0.0)
            m24 = m24.groupby(keys , sort = False, dropna = False).shift(1)
            s24 = s24.groupby(keys , sort = False, dropna = False).shift(1)

        c1 = c1.fillna(0.0).astype("float32")
        c24 = c24.fillna(0.0).astype("float32")
        m24 = m24.astype("float32")
        s24 = s24.replace(0, np.nan).fillna(1.0).astype("float32") 

        df_sorted.loc[valid_time, "txn_count_1h"] = c1.to_numpy(dtype = np.float32)
        df_sorted.loc[valid_time, "txn_count_24h"] = c24.to_numpy(dtype = np.float32)
        df_sorted.loc[valid_time, "amt_mean_24h"] = m24.to_numpy(dtype = np.float32)
        df_sorted.loc[valid_time, "amt_std_24h"] = s24.to_numpy(dtype = np.float32)

        z = (
            (df_sorted.loc[valid_time, "TransactionAmt"].astype("float32") - df_sorted.loc[valid_time, "amt_mean_24h"] ) / df_sorted.loc[valid_time, "amt_std_24h"]
        )

        z = z.replace([np.inf, -np.inf], 0.0).fillna(0.0).astype("float32")
        df_sorted.loc[valid_time, "amt_zscore"]  = z.to_numpy(dtype = np.float32)

   
    #def _roll(series, window):
       # return(
       #     series
        #    .groupby(df_sorted[entity_col])
         #   .rolling(window)
       # )
    
   

    df_out = df_sorted.sort_values("_row_id__", kind ="mergesort").reset_index(drop=True)
    df_out.drop(columns = ["_row_id__"], inplace = True)
    ##Abrupt change detector
    return df_out

      
    ###############
    #####Metrics####
def eval_metrics(y_true: np.ndarray, y_prob: np.ndarray):
        y_pred = (y_prob >= THRESH).astype(int)
        return {
            "AUC": float(roc_auc_score( y_true,  y_prob)) if len(np.unique(y_true)) > 1 else float("nan"),
            "f1":  float(f1_score(y_true, y_pred)) if len(np.unique(y_pred)) > 1 else float("nan"),
            "precision": float(precision_score(y_true, y_pred, zero_division = 0)),
            "recall" :  float(recall_score (y_true, y_pred, zero_division = 0)) ,
            } 
    
              
### FRAUD RING DETECTION ####
#################################
def plotAblation(csvPath, ttl="Experiment Ablation Results", sort="test_auc", topN=None):
    df = pd.read_csv(csvPath)
    if sort in df.columns:
        df = df.sort_values(sort, ascending=False)
    if topN is not None:
        df = df.head(topN)
    x = np.arange(len(df)) 
    w = 0.42
    plt.figure(figsize=(12,5))
    plt.bar(x - w/2, df["test_auc"].values, width=w, label="Test ROC-AUC")
    if "test_pr_auc" in df.columns:
        plt.bar(x + w/2, df["test_pr_auc"].values, width=w, label="Test PR-AUC")

    plt.xticks(x, df["experiment"].values, rotation = 30, ha="right")
    plt.ylabel("Metric")
    plt.title(ttl) 
    plt.legend()
    plt.tight_layout()
    plt.show()

def plotmyAdaptiveAlpha(alphacsv = "Myadaptivebudgetlearnedalpha.csv", ttl ="Adaptive alpha vs Budget plot"):
    df = pd.read_csv(alphacsv).sort_values("budget_K")
    plt.figure(figsize=(8, 4))
    plt.plot(df["budget_K"], df["alpha_star"], marker ="o", label ="alpha (learned)")
    plt.xlabel("Budget K")
    plt.ylabel("alpha")
    plt.ylim(-0.05, 1.05)
    plt.title(ttl)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
    if "gain" in df.columns:
        plt.figure(figsize=(8, 4))
        plt.bar(df["budget_K"].astype(str), df["gain"])
        plt.xlabel("Budget K")
        plt.ylabel("Precision gain(P@K learned fixed)")
        plt.title("Adaptive alpha vs benefit by Budget")
        plt.grid(True, axis="y",alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()




def detect_fraud_ring(data: HeteroData, labels, txn_id: int, hops: int = GRAPH_HOPS_FOR_RING) -> nx.Graph:
    # Optimize fraud ring detector
    import networkx as nx
   
    num_txn = data["transaction"].num_nodes
    if txn_id < 0 or txn_id >= num_txn:
        raise IndexError(f"txn_id {txn_id} out of bounds [0, {num_txn - 1}]")

    G = nx.Graph()  
    # Add main transaction node
    fraud_flag = int(labels[txn_id])
    G.add_node(f"txn_{txn_id}", type ="txn", fraud = fraud_flag)

    txn_to_attr = {}
    attr_to_txn = {}

    for (src, rel, dst) in data.edge_types:
        edge_index = data[(src, rel, dst)].edge_index
        
        if src == "transaction":
            txn_to_attr.setdefault(dst, {})
            t_list = edge_index[0].tolist()
            a_list = edge_index[1].tolist()
            
            for t, a in zip(t_list, a_list):
                txn_to_attr[dst].setdefault(t, []).append(a)

        if dst == "transaction":
            attr_to_txn.setdefault(src, {})
            a_list = edge_index[0].tolist()
            t_list = edge_index[1].tolist()

            for a, t in zip(a_list, t_list):
                attr_to_txn[src].setdefault(a, []).append(t)

     #--------- Multi hop BFS---------        
                
    frontier = {txn_id}
    visited_txn = {txn_id}
    visited_attr  = set()
    print("I am before hops")
    for _ in range(hops):
        next_frontier = set()
        
        for t in frontier:
            for attr_type, mapping in txn_to_attr.items():
                txns = frontier & mapping.keys() #set interaction
                for t in txns:                 
                    for attr_id in mapping[t]:
                        attr_node = f"{attr_type}_{attr_id}"
                    
                        if attr_node not in  visited_attr:
                            visited_attr.add(attr_node)
                            G.add_node(attr_node, type = attr_type)
                            G.add_edge(f"txn_{t}", attr_node)
                        
                        for t2  in attr_to_txn.get(attr_type, {}).get(attr_id, []):
                            if t2 not in visited_txn:
                                visited_txn.add(t2)
                                next_frontier.add(t2)
                                G.add_node(f"txn_{t2}" , type = "txn", fraud = int(labels[t2]))
                                G.add_edge(attr_node , f"txn_{t2}")
                      
                   
        frontier = next_frontier
    print("i am now at return G")
    return G       


def show_graph(G):
    #pos = nx.spring_layout(G , K=0.25)
    pos = nx.spring_layout(G , seed = SEED)
    colors = []
    for n in G.nodes:
        if G.nodes[n]["type"] == "txn":
            colors.append("red" if G.nodes[n].get("fraud", 0) == 1 else "lightgreen")
        else:
            colors.append("skyblue")

    plt.figure(figsize = (11,9)) 
    nx.draw(G, pos, with_labels = True, node_color = colors, node_size = 1200, font_size = 8)
    plt.title("Fraud ring Graph")
    #plt.tight_layout()
    plt.show()

def show_graph_interactive(G , output ="fraud_ring.html"):
    net = Network(height ="800px", width ="100%", notebook = False)
    net.toggle_physics(True)
    for n, attrs in G.nodes(data = True):
        color = "red" if attrs.get("fraud", 0) == 1 else "green"
        net.add_node(str(n), label = str(n), color = color)
    for u,v in G.edges():
        net.add_edge(str(u), str(v))
    net.write_html(output)
    print("Saved interactive graph")
    display(HTML(filename=output))


    
 ###### INFERENCE -SCORE NEW TRANSACTIONS ############

def score_new_transactions(model: nn.Module, new_df: pd.DataFrame) -> np.ndarray: 
    required = ["TransactionAmt", "TransactionDT"]
    for col in required:
        if col not in new_df.columns:
            raise ValueError(f"new_df missing required column: {col}")
            
    amt = new_df["TransactionAmt"].astype(np.float32).values
    tdt = new_df["TransactionDT"].astype(np.float32).values
    tdt_norm = normalizeArray(tdt)

    x = torch.tensor(np.stack([amt, tdt_norm], axis = 1), dtype = torch.float32, device = DEVICE)
    t = torch.tensor(tdt.reshape(-1,1), dtype = torch.float32, device = DEVICE)
 
    model.eval()
    with torch.no_grad():
        x_dict = {"transaction": x}
        edge_dict = {}
        logits = model(x_dict, edge_dict, t)
        probs = torch.sigmoid(logits).cpu().numpy()
        
    return probs

   ################################# 
    ##MAIN
   ###################################

def main():
    global POS_WEIGHT

    set_all_seeds(SEED)
    print(f"Device: {DEVICE}| AMP ={use_amp}")
  
    #filepath1 = 'bank_transactions_data_2.csv'
    filepath2 = 'train_transaction.csv'
    filepath3 = 'train_identity.csv'

    df1 = pd.read_csv(filepath2, nrows = TRAINROWS)
    df2 = pd.read_csv(filepath3, nrows = TRAINROWS)

    df = df1.merge(df2, on = "TransactionID", how ="left")
    df = reduce_mem_usage(df)
    ########################################################
    ### fill missing values for categorical fields
    cat_cols = ["DeviceInfo", "DeviceType", "id_30", "id_31", "P_emaildomain","addr1", "ProductCD","card1"]
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].fillna("unknown").astype(str)
        
    df["isFraud"] = df["isFraud"].fillna(0).astype(int)
    df = df.reset_index(drop = True)  ## keep consistent indexing 
    
    print("1. intial Column seen by df:", sorted(df.columns))

    ### Add Temporal behavior features
    print("\n[1] Add Temporal behavior features")
    df = add_temporal_behavior_features(df, entity_col ="card1", exclude_current=True)

    #print("2. Column seen by df:", sorted(df.columns))

    print("\n[2] Time based train/val/test split")
    train_idx, val_idx, test_idx,cutoff_train, cutoff_val= make_time_split_train_val_test(df, time_col="TransactionDT", train_val_frac= train_val_frac_time,
                                                                                          val_within_train= val_frac_within_train)
    print(f"[Split] cutoff_Train={cutoff_train: .2f} cutoff_val={cutoff_val:.2f} | "
          f"train ={len(train_idx):,} val ={len(val_idx):,} test={len(test_idx):,}")
    
    ### pos_weight from TRAIN only
    y_train = df.loc[train_idx.numpy(), "isFraud"].values.astype(np.float64)
    pos = max(1, int((y_train ==1 ).sum()))
    neg = max(1, int((y_train ==0 ).sum()))
    pw = neg / pos
    POS_WEIGHT = torch.tensor([pw], dtype=torch.float32, device=DEVICE)
    print(f"[Imbalance] train_pos ={pos:,} train_neg={neg:,} pos_weight={pw:.2f}")

    #Normalize time
    t_train = df.loc[train_idx.numpy(), "TransactionDT"].values.astype(np.float64)
    t_min, t_max = float(t_train.min()), float(t_train.max())
    print(f"[TimeNorm] train min={t_min:.2f} max={t_max:.2f}")

    print("\n[3]Building Hypergraph.......")  ### Million scale
    hg = DualHypergraph(df)
    
    #########################################################

    print("\n[4] ## build Hetero graph ##")
    hetero_full, node_sizes = build_hetero_graph(df, tdt_min= t_min, tdt_max=t_max)

    t_all = df["TransactionDT"].values.astype(np.float32)
    t_all_norm = ((t_all - t_min) / (t_max - t_min + 1e-6)).astype(np.float32)
    hetero_full["transaction"].time = torch.tensor(t_all_norm, dtype = torch.float32) 
    
    print("\n[5] Build leakage safe training Hetero Graph")
    hetero_train = filter_hetero_for_training(hetero_full, train_idx)
    ### Create train/test split for transactions 
    hetero_val = hetero_full
    hetero_test = hetero_full

    tx_feat_dim = hetero_full["transaction"].x.size(1)


    def make_mlp():
        return TxnMLP(in_dim = tx_feat_dim + 1, hidden=HIDDEN_DIM).to(DEVICE)
    
    def make_hyp():
        return HypergraphNN(
            num_nodes=hg.N,
            num_hyperedges=hg.total_hyperedges,
            K=hg.K,
            dim=ENTITYDIM,
            hidden=HIDDEN_DIM,
            heads=4
        ).to(DEVICE)

    def make_het(use_time =True):
        m = HeteroHGTModel(
            hetero_full.metadata(),
            node_sizes=node_sizes,
            tx_feat_dim=tx_feat_dim,
            hidden=HIDDEN_DIM,
            time_dim=TIME_DIM,
            heads=4,
            use_time=use_time,
            dropout_p=0.2
        ).to(DEVICE)
        m.transaction_time = hetero_full["transaction"].time.to(DEVICE)
        return m
    
    def make_head():
        return BinaryHead(hidden_dim=HIDDEN_DIM).to(DEVICE)
    
    def make_fusion():
        return FusionGated(hidden_dim= HIDDEN_DIM, dropout_p=0.1).to(DEVICE)
        #return FusiontestFinal(hidden_dim= HIDDEN_DIM).to(DEVICE)
    
    def make_fusion_analsis():
        return FusiontestFinal(hidden_dim= HIDDEN_DIM).to(DEVICE)
    
  
    def make_time_enc():
        te = Time2VecTxnEncoder(tx_feat_dim= tx_feat_dim, hidden= HIDDEN_DIM, time_dim= TIME_DIM).to(DEVICE)
        te._global_time = hetero_full["transaction"].time.to(DEVICE)
        return te

    #### Ablation experiments
    experiments = [
        ("TXN-MLP(feature only)", {
            "mlp": make_mlp(),"mlp_time":True,"_global_time": hetero_full["transaction"].time.to(DEVICE),
            "hyp": None, "het": None, "head": None, "fusion": None, "time_enc": None}),
        ("Heterogeneous(no time)", {
            "mlp": None,"mlp_time":False,"_global_time": None, 
             "hyp":None, "het": make_het(use_time=False), "head": make_head(), "fusion": None, "time_enc": None}),
        ("Heterogeneous + Time2Vec", {
            "mlp": None,"mlp_time":False,"_global_time": None, 
            "hyp":None, "het": make_het(use_time=True), "head": make_head(), "fusion": None, "time_enc": None}),
        ("Hypergraph-only", {
            "mlp": None,"mlp_time":False,"_global_time": None, 
             "hyp":make_hyp(), "het": None, "head": make_head(), "fusion": None, "time_enc": None}),
       # ("FusionAnalysis(Hyper+ Time2Vec)", {
          #  "mlp": None,"mlp_time":False,"_global_time": None, 
           # "hyp":make_hyp(), "het": None, "head": None, "fusion": make_fusion_analsis(), "time_enc":make_time_enc()}),
        ("FusionAnalysis(Hyper+ Heterogeneous + Time2Vec)", {
            "mlp": None,"mlp_time":False,"_global_time": None, 
            "hyp":make_hyp(), "het": make_het(use_time=True), "head": None, "fusion": make_fusion_analsis(), "time_enc": None}),
        ("FusionAnalysis(Hyper+ Heterogeneous + no time)", {
            "mlp": None,"mlp_time":False,"_global_time": None, 
            "hyp":make_hyp(), "het": make_het(use_time=False), "head": None, "fusion": make_fusion_analsis(), "time_enc": None})
        #("FusionAnalysis(Hyper+ Hetero-HGT + Time2Vec+FusionTime)", {
            #"mlp": None,"mlp_time":False,"_global_time": None, 
            #"hyp":make_hyp(), "het": make_het(use_time=True), "head": None, "fusion": make_fusion_analsis(), "time_enc": make_time_enc()})    
        
    ]

    results = []

    full_fusion_bundle = None

    for exp_name, comps in experiments:
        best_val_auc  = train_experiment(exp_name, comps, hg, hetero_train, train_idx, hetero_val, val_idx)
        
        train_met, _, _ = score_experiment(exp_name, comps, hg, hetero_train, train_idx)
        val_met, yv, pv = score_experiment(exp_name, comps, hg,hetero_val, val_idx)
        test_met, yt, pt = score_experiment(exp_name, comps, hg, hetero_test, test_idx)

        if exp_name == "FusionAnalysis(Hyper+ Heterogeneous + Time2Vec)":
            full_fusion_bundle = {
                "hyp": comps["hyp"],
                "het": comps["het"],
                "fusion": comps["fusion"]
            }
            print("[RingStage] Stored Full fusion for offline ring analysis")

        print(f"[FINAL] {exp_name} | train_auc={train_met['auc']:.4f}"
              f"val_auc = {val_met['auc']:.4f} test_auc={test_met['auc']:.4f} (best_val_auc={best_val_auc:.4f})")

        plot_confusion_matrix(yt, pt, threshold= THRESH, title = f"CM(Test): {exp_name}")
        results.append({"experiment": exp_name, 
                        "train_auc": train_met["auc"],
                        "val_auc": val_met["auc"],
                        "test_auc": test_met["auc"],
                        "test_pr_auc": test_met["pr_auc"],
                        "best_val_auc": float(best_val_auc)
                        
                        })
        
       
        ### XGBoost ##############
    print("##XGBoost Analysis start*****")
    xgb_met = run_xgboost_baseline(df, train_idx, test_idx, t_min, t_max, threshold=THRESH)
    if xgb_met is not None:
        results.append({"experiment": "XGBoost(tabular baseline)",
                        "train_auc": float("nan"),
                        "val_auc": float("nan"),
                        "test_auc": xgb_met["auc"],
                        "test_pr_auc": xgb_met["pr_auc"],
                        "best_val_auc": float("nan")
                        })   

    res_df = pd.DataFrame(results).sort_values("test_auc", ascending = False, na_position="last").reset_index(drop= True)
    print("\n################## SUMMARY (sorted by Test AUC)    ################")
    print(res_df.to_string(index= False)) 
    res_df.to_csv(out_ablation_csv, index= False) 
    plotAblation("ablationresults.csv",ttl="FusionAnalysis Ablations", sort="test_auc") 
       

    ### offline ring stage ###

    if RUN_RING_STAGE:
        if full_fusion_bundle is None:
            print("[RingStage] Full fusion model not trained. Skipping")
            return

        print("\n =================== OFFLINE RING STAGE(FULL FUSION ONLY ================")
        hyp = full_fusion_bundle["hyp"]
        het = full_fusion_bundle["het"]
        fusion = full_fusion_bundle["fusion"] 

        labels_global = hetero_full["transaction"].y.cpu().numpy().astype(int)
        txn_time_global = df["TransactionDT"].values.astype(np.float64)

        test_ids_np, probs_np, trustNp = score_full_fusion_on_test(hg, hetero_full, test_idx, hyp, het, fusion)

        df_ring, queue_df, comp_df, adaptive_df  = run_offline_ring_stage(
            test_ids_np=test_ids_np,
            probs_np=probs_np,
            trustNp = trustNp,
            labels_global=labels_global,
            txn_time_global=txn_time_global,
            hetero_full=hetero_full
        )

        if df_ring is not None:
            df_ring.to_csv(out_ring_metrics_csv, index = False)
            print(f"Saved: {out_ring_metrics_csv}")
        if queue_df is not None:
            queue_df.to_csv(out_queue_csv, index = False) 
            print(f"Saved: {out_queue_csv}") 

        if comp_df is not None:
            comp_df.to_csv(out_budget_csv, index = False) 
            print(f"Saved: {out_budget_csv}")   
        if adaptive_df is not None:
            adaptive_df.to_csv(out_adaptive_budget_csv, index = False)
            print(f"Saved: {out_adaptive_budget_csv}")
            plotmyAdaptiveAlpha("Myadaptivebudgetlearnedalpha.csv", ttl="AdaptiveBudgetOptimizer")  

     
   ########################################################
      
                

if __name__ == "__main__":
    print("****start My analysis*****")
    print("current working directory:", os.getcwd())
    #print("Files here", os.listdir(os.getcwd()))
    main()
    
